# 🟢 dorina-zero: Othello (8x8) AlphaZero Colab Eğitimi

Bu notebook, **dorina-zero** genel amaçlı AlphaZero motorunun Google Colab ortamında (GPU + High-RAM/CPU) çalıştırılması için optimize edilmiştir.

### Özellikler:
- **Oyun:** 8x8 Othello (Reversi)
- **Veri Artırımı:** 8x D4 Dihedral Simetrisi
- **Motor:** Generic MCTS (200 simülasyon) + OthelloNet (4 ResBlock, ~323K Parametre)
- **Gating:** %55 eşik, tarafsız açılışlar, Historical Rakip desteği

In [ ]:
# 1. (İsteğe Bağlı) Google Drive Bağlantısı — Checkpoint'lerin kalıcı olması için
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Depoyu çekin (veya en güncel commit'i güncelleyin)
!git clone https://github.com/atalhatulu/dorina-zero.git
%cd dorina-zero
!git pull

In [ ]:
# 3. Gerekli kütüphaneleri kurun
!pip install -q -r requirements.txt
!pip install -q pytest

In [ ]:
# 4. Donanım ve Ortam Doğrulaması
import os, torch
print(f"PyTorch Versiyonu: {torch.__version__}")
print(f"CUDA Aktif mi?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CPU Çekirdek Sayısı: {os.cpu_count()}")

# Test suite'i çalıştırıp doğrula
!pytest tests/test_othello.py

In [ ]:
# 5. Othello AlphaZero Eğitimini Başlatın (6. İterasyondan Devam veya Sıfırdan)
# Colab CPU çekirdek sayısına göre workers otomatik ayarlanır
import os
num_workers = max(1, (os.cpu_count() or 4) - 1)

!python train_generic.py \
  --game othello \
  --iterations 15 \
  --games-per-iter 50 \
  --sims 200 \
  --epochs 3 \
  --batch-size 64 \
  --lr 1.5e-4 \
  --workers {num_workers} \
  --dirichlet-alpha 0.3 \
  --historical-ratio 0.20 \
  --gating-every 1 \
  --gating-games 24 \
  --gating-sims 150 \
  --gating-threshold 0.55 \
  --out-dir checkpoints/othello \
  --resume checkpoints/othello/current_best.pt

In [ ]:
# 6. (Opsiyonel) Eğitim Checkpoint'lerini Google Drive'a Yedekleyin
!cp -r checkpoints/othello /content/drive/MyDrive/dorina_checkpoints_othello
print("Checkpoint'ler Google Drive'a yedeklendi!")